# splicetarget — Real Data Walkthrough

End-to-end analysis of patient long-read RNA sequencing data for aberrant splicing
characterization and ASO therapeutic target nomination.

This notebook walks through every pipeline stage using the **Python API** on real
PacBio IsoSeq or ONT direct-RNA data. Each stage produces structured Python objects
that feed directly into the next.

---

### Requirements

| Dependency | Purpose |
|---|---|
| `splicetarget` | `pip install -e ".[dev]"` |
| `minimap2 ≥ 2.26` | Splice-aware long-read alignment |
| `samtools ≥ 1.18` | BAM sorting/indexing |

### Input Files

| File | Example |
|---|---|
| Long reads | `patient_isoseq.bam` or `.fastq.gz` (PacBio / ONT) |
| Reference genome | `GRCh38.fa` + `.fai` index |
| Gene annotation | `gencode.v44.annotation.gtf` |
| Transcriptome (optional) | `gencode.v44.transcripts.fa` for off-target screening |

---
## 0 — Configuration

Set your file paths below. Everything else flows from these.

In [ ]:
from pathlib import Path

# ── USER CONFIG — edit these paths ──
READS       = Path("patient_isoseq.bam")                   # aligned BAM or raw FASTQ
REFERENCE   = Path("GRCh38.fa")                            # indexed reference genome
ANNOTATION  = Path("gencode.v44.annotation.gtf")           # GENCODE GTF
GENE        = "DMD"                                        # target gene symbol
OUTDIR      = Path(f"results/{GENE}")

# Optional
TRANSCRIPTOME = None   # Path("gencode.v44.transcripts.fa") for off-target BLAST
READ_TYPE     = "isoseq"   # isoseq | pb_cdna | ont_drna | ont_cdna
THREADS       = 8
MIN_READS     = 2
JUNCTION_TOL  = 10

OUTDIR.mkdir(parents=True, exist_ok=True)
print(f"Gene:       {GENE}")
print(f"Reads:      {READS}")
print(f"Reference:  {REFERENCE}")
print(f"Annotation: {ANNOTATION}")
print(f"Output:     {OUTDIR}")

In [ ]:
from __future__ import annotations

import json
import time
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Markdown, display

# ── splicetarget modules ──
from splicetarget.data.io import iter_aligned_reads, extract_splice_junctions
from splicetarget.data.reference import ReferenceTranscriptome
from splicetarget.alignment.aligner import ReadType, align_long_reads, compute_alignment_stats
from splicetarget.isoforms.collapse import collapse_isoforms
from splicetarget.isoforms.classify import classify_isoforms, IsoformCategory
from splicetarget.isoforms.quantify import quantify_isoforms
from splicetarget.splicing.events import detect_aberrant_events, EventType
from splicetarget.therapeutic.aso_design import design_aso_candidates, DesignParams
from splicetarget.therapeutic.scoring import ScoringWeights, rescore_candidates, candidates_to_dataframe
from splicetarget.visualization.sashimi import plot_sashimi
from splicetarget.utils.genome import gc_content, reverse_complement

plt.rcParams.update({
    "figure.dpi": 140, "font.size": 10,
    "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})

PALETTE = {
    "FSM": "#27AE60", "ISM": "#2ECC71", "NIC": "#F39C12",
    "NNC": "#E74C3C", "IR": "#E67E22", "GG": "#95A5A6",
}

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
t0 = time.time()
print("splicetarget loaded ✓")

---
## 1 — Alignment (skip if BAM already aligned)

If your input is already an **aligned, indexed BAM**, this stage is skipped automatically.  
If starting from raw FASTQ, `minimap2` aligns in splice-aware mode with the correct preset for your platform.

In [ ]:
if READS.suffix in (".bam", ".cram"):
    aligned_bam = READS
    print(f"Input is aligned BAM — skipping alignment.")
else:
    aligned_bam = OUTDIR / f"{GENE}_aligned.bam"
    print(f"Aligning with minimap2 (preset: {READ_TYPE}, threads: {THREADS})...")
    stats = align_long_reads(
        reads_path=READS,
        reference_path=str(REFERENCE),
        output_bam=aligned_bam,
        read_type=ReadType(READ_TYPE),
        threads=THREADS,
    )
    print(f"Alignment complete → {aligned_bam}")

# QC stats
aln_stats = compute_alignment_stats(str(aligned_bam))
print(f"\nAlignment QC:")
print(f"  Total reads:   {aln_stats.total_reads:,}")
print(f"  Mapped:        {aln_stats.mapped_reads:,} ({aln_stats.mapping_rate:.1%})")
print(f"  Spliced:       {aln_stats.spliced_reads:,} ({aln_stats.splice_rate:.1%} of mapped)")
print(f"  Median length: {aln_stats.median_read_length:,} bp")

---
## 2 — Load Reference Transcriptome

Parse the GENCODE annotation into an SQLite-backed database (cached after first run).  
Extract the gene model with all known transcript isoforms.

In [ ]:
ref_db = ReferenceTranscriptome(ANNOTATION)
gene_model = ref_db.get_gene(GENE)

assert gene_model is not None, f"Gene '{GENE}' not found in {ANNOTATION}"

canonical = gene_model.get_canonical_transcript()

print(f"Gene:        {gene_model.gene_name} ({gene_model.gene_id})")
print(f"Locus:       {gene_model.chrom}:{gene_model.start:,}-{gene_model.end:,} ({gene_model.strand})")
print(f"Biotype:     {gene_model.biotype}")
print(f"Transcripts: {gene_model.transcript_count}")
if canonical:
    print(f"Canonical:   {canonical.transcript_id} ({canonical.exon_count} exons)")
print(f"\nReference splice junctions (union): {len(ref_db.get_all_splice_junctions(GENE))}")

---
## 3 — Extract Reads in Gene Region

Pull all aligned reads overlapping the gene locus from the BAM.  
Optionally filter by mapping quality and require at least one splice junction.

In [ ]:
region = f"{gene_model.chrom}:{gene_model.start}-{gene_model.end}"

reads = list(iter_aligned_reads(
    str(aligned_bam),
    region=region,
    min_mapq=20,
    require_splice=True,
))

print(f"Region: {region}")
print(f"Reads extracted: {len(reads):,}")
if reads:
    lengths = [r.length for r in reads]
    exon_counts = [len(r.exon_blocks) for r in reads]
    print(f"Read length:  median {np.median(lengths):,.0f} bp, range {min(lengths):,}–{max(lengths):,} bp")
    print(f"Exons/read:   median {np.median(exon_counts):.0f}, range {min(exon_counts)}–{max(exon_counts)}")

---
## 4 — Isoform Collapsing

Group reads by splice junction chain to identify unique transcript isoforms.  
Reads with identical intron chains (within `JUNCTION_TOL` bp wobble) are merged.

In [ ]:
isoforms = collapse_isoforms(
    reads,
    junction_tolerance=JUNCTION_TOL,
    end_tolerance=50,
    min_reads=MIN_READS,
    gene_name=GENE,
)

iso_df = pd.DataFrame([{
    "Isoform": iso.isoform_id,
    "Exons": iso.exon_count,
    "Reads": iso.read_count,
    "Exon bp": f"{iso.total_exon_length:,}",
    "Span": f"{iso.chrom}:{iso.start:,}-{iso.end:,}",
} for iso in isoforms])

print(f"Collapsed {len(reads):,} reads → {len(isoforms)} unique isoforms (min_reads={MIN_READS})\n")
display(iso_df)

---
## 5 — Isoform Classification (SQANTI3-style)

Classify each patient isoform against the reference gene model:

| Category | Meaning | Therapy-relevant? |
|---|---|---|
| **FSM** | Full Splice Match — all junctions match a known transcript | No |
| **ISM** | Incomplete Splice Match — junction subset of known transcript | No |
| **NIC** | Novel In Catalog — new junction combo from known donors/acceptors | Sometimes |
| **NNC** | Novel Not in Catalog — at least one novel splice site | **Yes** |
| **IR** | Intron Retention — exon block spans a reference intron | **Yes** |

In [ ]:
classified = classify_isoforms(isoforms, gene_model, junction_tolerance=JUNCTION_TOL)

class_df = pd.DataFrame([{
    "Isoform": c.isoform.isoform_id,
    "Category": c.category.value,
    "Ref Transcript": c.best_ref_transcript or "—",
    "Junction Match": f"{c.junction_match_pct:.0%}",
    "Reads": f"{c.isoform.read_count:,}",
    "Therapy Candidate": "✓" if c.is_therapy_candidate else "",
} for c in classified])

print(f"{len(classified)} isoforms classified\n")
display(class_df)

---
## 6 — Expression Quantification

In [ ]:
expression = quantify_isoforms(classified, total_gene_reads=len(reads))

expr_df = pd.DataFrame([{
    "Isoform": e.isoform_id,
    "Category": e.category,
    "Reads": f"{e.read_count:,}",
    "Fraction": f"{e.fraction:.1%}",
    "TPM est.": f"{e.tpm_estimate:,.0f}",
    "Aberrant": "✓" if e.is_aberrant else "",
} for e in expression])

display(expr_df)

In [ ]:
# ── Dual panel figure: classification pie + expression bar ──
fig, (ax_pie, ax_bar) = plt.subplots(1, 2, figsize=(13, 4.5),
                                      gridspec_kw={"width_ratios": [1, 1.4]})

# Pie chart — isoform categories weighted by read count
cat_reads = {}
for c in classified:
    cat = c.category.value
    cat_reads[cat] = cat_reads.get(cat, 0) + c.isoform.read_count

labels, sizes, pie_colors = [], [], []
for cat in ["FSM", "ISM", "NIC", "NNC", "IR", "GG", "GI"]:
    if cat in cat_reads:
        labels.append(cat)
        sizes.append(cat_reads[cat])
        pie_colors.append(PALETTE.get(cat, "#BDC3C7"))

ax_pie.pie(sizes, labels=labels, colors=pie_colors,
           autopct=lambda p: f"{p:.1f}%" if p > 3 else "",
           startangle=140, wedgeprops={"edgecolor": "white", "linewidth": 1.5})
ax_pie.set_title("Isoform Classification (by reads)", fontweight="bold")

# Bar chart — expression per isoform
expr_sorted = sorted(expression, key=lambda e: e.fraction)
ax_bar.barh(
    [e.isoform_id for e in expr_sorted],
    [e.fraction * 100 for e in expr_sorted],
    color=[PALETTE.get(e.category, "#BDC3C7") for e in expr_sorted],
    edgecolor="white", height=0.65,
)
ax_bar.set_xlabel("% of Gene Expression")
ax_bar.set_title("Isoform Expression Profile", fontweight="bold")
for i, e in enumerate(expr_sorted):
    ax_bar.text(e.fraction * 100 + 0.3, i, f"{e.read_count:,}",
               va="center", fontsize=8, color="#555")

seen = set()
handles = []
for e in expression:
    if e.category not in seen:
        seen.add(e.category)
        handles.append(mpatches.Patch(color=PALETTE.get(e.category, "#BDC3C7"), label=e.category))
ax_bar.legend(handles=handles, fontsize=9, loc="lower right")

plt.tight_layout()
fig.savefig(OUTDIR / f"{GENE}_classification_expression.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

---
## 7 — Aberrant Splicing Event Detection

Compare each novel isoform to the reference to identify clinically actionable splicing defects:

- **Exon Skipping** — one or more exons excluded
- **Cryptic Exon** — novel exonic sequence included from intronic regions
- **Intron Retention** — failure to splice out an intron
- **Alt 5'/3' SS** — shifted splice donor or acceptor site

In [ ]:
events = detect_aberrant_events(
    classified,
    gene_model,
    total_gene_reads=len(reads),
    min_read_support=MIN_READS,
    junction_tolerance=JUNCTION_TOL,
)

if events:
    event_df = pd.DataFrame([{
        "Event": ev.event_id,
        "Type": ev.event_type.value,
        "Region": f"{ev.chrom}:{ev.start:,}-{ev.end:,}",
        "Reads": f"{ev.total_read_support:,}",
        "Gene %": f"{ev.fraction_of_gene_expression:.1%}",
        "Frame": ev.reading_frame_impact or "unknown",
        "ASO target?": "✓" if ev.is_aso_target else "✗",
        "Summary": ev.summary(),
    } for ev in events])
    print(f"{len(events)} aberrant splicing event(s) detected\n")
    display(event_df)
else:
    print("No aberrant splicing events above threshold.")
    print("All isoforms match known reference transcripts (FSM/ISM).")

---
## 8 — ASO Candidate Design & Scoring

For each ASO-amenable event, generate candidate antisense oligonucleotide binding sites:  
- 18–25 nt sliding window across the target region  
- Multi-criteria scoring: GC content, self-complementarity, splice proximity, length, accessibility  
- Optionally BLAST against the transcriptome for off-target specificity

In [ ]:
# Design candidates (requires indexed reference FASTA)
candidates = design_aso_candidates(events, str(REFERENCE))

if candidates:
    print(f"{len(candidates)} ASO candidates generated\n")

    # Off-target assessment (if transcriptome provided)
    if TRANSCRIPTOME:
        from splicetarget.therapeutic.offtarget import assess_off_targets
        assess_off_targets(candidates, str(TRANSCRIPTOME))
        candidates = rescore_candidates(candidates)
        print(f"Off-target BLAST complete against {TRANSCRIPTOME}")
    else:
        candidates = rescore_candidates(candidates)
        print("(Off-target assessment skipped — no transcriptome FASTA provided)")

    # Ranked table
    result_df = candidates_to_dataframe(candidates)
    display(result_df.head(20))

    # Save full table
    csv_path = OUTDIR / f"{GENE}_aso_candidates.csv"
    result_df.to_csv(csv_path, index=False)
    print(f"\nFull table saved → {csv_path}")
else:
    print("No ASO candidates generated (no ASO-amenable events detected).")

In [ ]:
# ── ASO ranking visualization ──
if candidates:
    top = candidates[:15]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(4, 0.4 * len(top))),
                                    gridspec_kw={"width_ratios": [1, 1.6]})

    ids = [c.candidate_id for c in top]
    scores = [c.composite_score for c in top]

    # Left: composite ranking
    ax1.barh(ids, scores,
             color=["#1ABC9C" if s >= 0.6 else "#E0E0E0" for s in scores],
             edgecolor="white", height=0.6)
    ax1.axvline(0.6, color="#E74C3C", ls="--", lw=1.2, label="High-confidence threshold")
    ax1.set_xlabel("Composite Score")
    ax1.set_title("ASO Candidate Ranking", fontweight="bold")
    ax1.invert_yaxis()
    ax1.legend(fontsize=8)

    # Right: stacked component breakdown
    components = {
        "GC":         [c.gc_score * 0.20 for c in top],
        "Self-comp":  [c.self_comp_score * 0.15 for c in top],
        "Proximity":  [c.splice_proximity_score * 0.20 for c in top],
        "Length":     [c.length_score * 0.10 for c in top],
        "Access":     [c.accessibility_score * 0.15 for c in top],
        "Off-target": [max(0, 1 - c.off_target_hits / 5) * 0.20
                       if c.off_target_hits >= 0 else 0.10 for c in top],
    }
    comp_colors = ["#3498DB", "#9B59B6", "#E74C3C", "#F39C12", "#1ABC9C", "#2C3E50"]
    left = np.zeros(len(top))
    for (name, vals), color in zip(components.items(), comp_colors):
        ax2.barh(ids, vals, left=left, color=color, edgecolor="white", height=0.6, label=name)
        left += np.array(vals)
    ax2.set_xlabel("Weighted Contribution")
    ax2.set_title("Score Component Breakdown", fontweight="bold")
    ax2.invert_yaxis()
    ax2.legend(fontsize=7, ncol=3, loc="lower right")

    plt.tight_layout()
    fig.savefig(OUTDIR / f"{GENE}_aso_ranking.png", dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()

---
## 9 — Sashimi Visualization

Publication-quality gene model showing reference transcript, patient isoforms
(color-coded by SQANTI3 category), aberrant events, and ASO target sites.

In [ ]:
plot_path = OUTDIR / f"{GENE}_sashimi.png"

fig = plot_sashimi(
    gene=gene_model,
    classified_isoforms=classified,
    events=events if events else None,
    aso_candidates=candidates[:10] if candidates else None,
    output_path=plot_path,
    title=f"{GENE} — Patient Splice Isoform Analysis",
)

print(f"Saved → {plot_path}")
plt.show()

---
## 10 — Clinical Summary Report

In [ ]:
total_reads = len(reads)
aberrant_isos = [c for c in classified if c.category.is_aberrant]
aberrant_reads = sum(c.isoform.read_count for c in aberrant_isos)
aberrant_pct = aberrant_reads / total_reads * 100 if total_reads else 0
n_therapy = sum(1 for c in classified if c.is_therapy_candidate)
n_events = len(events) if events else 0
n_aso = len(candidates) if candidates else 0
high_conf = sum(1 for c in candidates if c.is_high_confidence) if candidates else 0

report = f"""
### Clinical Summary Report

| Metric | Value |
|--------|-------|
| **Gene** | {gene_model.gene_name} ({gene_model.chrom}:{gene_model.start:,}–{gene_model.end:,}) |
| **Total reads in locus** | {total_reads:,} |
| **Unique isoforms** | {len(classified)} (min {MIN_READS} reads) |
| **Known isoforms (FSM/ISM)** | {sum(1 for c in classified if c.category.is_known)} |
| **Novel isoforms (NIC/NNC/IR)** | {sum(1 for c in classified if c.category.is_novel)} |
| **Aberrant expression** | {aberrant_pct:.1f}% of gene reads |
| **Aberrant events detected** | {n_events} |
| **Therapy-candidate isoforms** | {n_therapy} |
| **ASO candidates generated** | {n_aso} |
| **High-confidence ASOs** | {high_conf} (score ≥ 0.6, off-targets ≤ 3) |

**Interpretation:** {aberrant_pct:.0f}% of gene expression derives from aberrant
splice forms ({', '.join(c.category.value for c in aberrant_isos) or 'none'}).
{n_events} actionable splicing event(s) were detected. {n_aso} ASO candidates
were nominated, of which {high_conf} met the high-confidence threshold for
experimental validation.
"""

display(Markdown(report))

---
## 11 — Save Results

In [ ]:
elapsed = time.time() - t0

# JSON summary
results = {
    "gene": gene_model.gene_name,
    "gene_id": gene_model.gene_id,
    "region": f"{gene_model.chrom}:{gene_model.start}-{gene_model.end}",
    "total_reads": total_reads,
    "unique_isoforms": len(classified),
    "classification": {c.category.value: sum(1 for x in classified if x.category == c.category)
                       for c in classified},
    "aberrant_expression_pct": round(aberrant_pct, 2),
    "events_detected": n_events,
    "aso_candidates": n_aso,
    "high_confidence_asos": high_conf,
    "runtime_seconds": round(elapsed, 1),
}

json_path = OUTDIR / f"{GENE}_results.json"
with open(json_path, "w") as f:
    json.dump(results, f, indent=2, default=str)

print(f"Pipeline complete in {elapsed:.1f}s")
print(f"\nOutputs in {OUTDIR}/")
for p in sorted(OUTDIR.glob(f"{GENE}_*")):
    print(f"  {p.name}  ({p.stat().st_size / 1024:.1f} KB)")

---

## CLI Equivalent

Everything above can be run as a single command:

```bash
splicetarget run \
    --reads patient_isoseq.bam \
    --reference GRCh38.fa \
    --annotation gencode.v44.annotation.gtf \
    --gene DMD \
    --outdir results/DMD/ \
    --read-type isoseq \
    --threads 8 \
    --min-reads 2
```

See `splicetarget run --help` for all options.